# RA Codes and Belief Propagation: A Short Introduction

This notebook connects the equations to a working Rust decoder. We start with a four-bit code, inspect its parity-check matrix, perform systematic encoding, and examine a decoding result.

**Key idea:** `k` information bits are mapped to a length-`n` codeword. The added bits provide parity checks. A receiver uses these checks and probabilistic information about the channel to correct errors.


In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd().resolve()
while not (ROOT / "Cargo.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if not (ROOT / "Cargo.toml").exists():
    raise RuntimeError("Open this notebook from inside the ra-rust project")
sys.path.insert(0, str(ROOT / "python"))
from ra_reconcile import decode, decode_batch, encode
MATRIX_SMALL = ROOT / "data" / "H_small.edges"
MATRIX_RA = ROOT / "data" / "RA_n6000_R150.edges"


## Terms and Equations

`H` is a binary parity-check matrix and `c` is a codeword. A valid codeword satisfies `H cᵀ = 0 (mod 2)`. The result `H cᵀ` is called the **syndrome**: a zero syndrome means every parity check passed.

In a systematic RA code, `H=[A|T]`, where `T` is a lower double-diagonal matrix. For information bits `u`, the parity bits are `v₀ = XOR(A₀·u)` and `vᵢ = vᵢ₋₁ XOR XOR(Aᵢ·u)`. The codeword is `c=(u,v)`, the message length is `k=n−m`, and the code rate is `R=k/n`.

For a BSC with error probability `p`, the channel LLR is `Lⱼ=(1−2zⱼ) ln((1−p)/p)`, where `z=A XOR M`. A positive LLR favors bit 0; a negative LLR favors bit 1. Belief propagation (BP) exchanges these soft estimates between bit and check nodes until the syndrome is zero or the iteration limit is reached.


In [ ]:
# The matrix has two check nodes and four bit nodes.
print(MATRIX_SMALL.read_text())
k, n = 2, 4
print(f"Information bits k={k}, code bits n={n}, rate R=k/n={k/n:.2f}")

## Encoding by Hand and with the CLI

The small graph has checks `c₀ XOR c₁ XOR c₂ = 0` and `c₁ XOR c₂ XOR c₃ = 0`. For message `10`, the accumulator produces parity bits `11`, so the codeword is `1011`.


In [ ]:
word = encode(MATRIX_SMALL, "10")
print("encoded codeword:", word)
assert word == "1011"

## A Decoding Example

In this fixed test, Alice has `A=0110`, the public message is `M=1000`, and the estimated channel error probability is `p=0.1`. The combined observation `A XOR M=1110` differs by one bit from the valid word `0110`. BP uses the parity checks to recover the word.


In [ ]:
result = decode(MATRIX_SMALL, alice="0110", message="1000", p=0.1,
                max_iter=10, threads=1)
print(result)
print("estimate:", result.estimate, "syndrome passed:", result.success)
assert result.estimate == "0110" and result.success

## Keep in Mind

An all-zero syndrome does not prove that the decoded word is the transmitted word: the decoder may converge to a different valid codeword. Public messages also reveal some information about the key. A complete QKD protocol must account for this leakage and perform privacy amplification.

The next notebooks explore the effect of `p` and Monte Carlo sampling uncertainty.
